<a href="https://colab.research.google.com/github/YOUR_GITHUB_USER/YOUR_REPO/blob/main/track3_fintech_moneymate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 💳 Track 3 · Fintech
## MoneyMate: AI Spending Insights, Fraud Alerts & Money Coach

**Coaching session · ~60 min**

### The real-world problem
Banking apps show a long list of cryptic lines like `GRAB *TRANSPORT`, `SP SERVICES`, `ONLINE STORE XJ7`. Customers:
- don't know where their money goes each month,
- miss fraud signs, such as small "test" charges before a big one, or duplicate charges, and
- get generic, unhelpful alerts (*"Unusual activity detected"*) that they ignore.

Banks want **personalised insight** and **better fraud detection** without a human analyst per customer.

### The fintech angle
Combine **classic machine learning** (anomaly detection on numbers) with **generative AI** (understanding merchant names and explaining in plain English). This hybrid pattern is how many real banking AI features are built.

### What you will build
| Step | Technique | What it does |
|---|---|---|
| Data | `pandas` + synthetic generator | 3 months of realistic card transactions, with hidden fraud |
| Categorise | **Batch structured output** | 1 LLM call labels every merchant (not 1 call per row!) |
| Detect | **Isolation Forest + rules** | Classic ML finds the weird transactions |
| Explain | **LLM explanations** | Friendly, specific fraud alerts with a recommended action |
| Coach | **"pandas calculates, LLM narrates"** | Chat about your spending, grounded in real numbers |

## Step 1: Install Dependencies
- `google-genai`: Google's Gen AI SDK for Gemini (already preinstalled in Colab).
- `gradio`: the web UI for the demo.

`pandas`, `scikit-learn` and `matplotlib` are already in Colab.

The first line records the versions of a few core packages Colab depends on, so the install below doesn't upgrade them and break Colab. If Colab asks you to **restart the session** after installing, click restart and continue from Step 2.

In [ ]:
# Keep Colab's own versions of these packages to avoid dependency conflicts
!pip freeze | grep -iE '^(google-auth|opentelemetry-(api|sdk))==' > /tmp/colab-constraints.txt
!pip install -q -c /tmp/colab-constraints.txt google-genai gradio==6.29.1

## Step 2: Setup Google Gemini API Key
1. Go to [Google AI Studio](https://aistudio.google.com/).
2. Sign in with your Google account and click **Get API key**.
3. In Colab, click the 🔑 **Secrets** icon on the left sidebar.
4. Create a new secret named `GEMINI_API_KEY` and paste your key.
5. Turn on the **Notebook access** toggle for that secret.

We also define `ask_gemini()`, a small wrapper that waits and retries when the free tier says *"429: too many requests"*. Free-tier keys only allow a few requests per minute, so every team will hit this during the hackathon.

In [ ]:
import time
from google import genai
from google.genai import types
from google.colab import userdata

# Securely load the API key from Colab Secrets
try:
    GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=GOOGLE_API_KEY)
    print("✅ Gemini API Key configured successfully!")
except Exception as e:
    print("❌ Error: Please ensure you added 'GEMINI_API_KEY' to your Colab Secrets and turned on 'Notebook access'.")
    print("   Details:", e)
    raise

# One place to change the model for the whole notebook
MODEL = "gemini-2.5-flash"

def ask_gemini(contents, config=None, retries=3):
    """Call Gemini, waiting and retrying if we hit the free-tier rate limit."""
    for attempt in range(retries):
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=config)
        except Exception as e:
            if "429" in str(e) and attempt < retries - 1:
                wait = 20 * (attempt + 1)
                print(f"⏳ Rate limited by the free tier, retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise

## Step 3: Generate Realistic Transactions
Real bank data is private, so hackathon teams use **synthetic data**. We simulate one person's card for July to September 2026: coffee, groceries, rides, bills and salary.

We also secretly plant **4 fraud patterns** that real fraud teams look for:
1. 💎 a large overseas purchase at 2:47 am,
2. 🧪 *card testing*: five $1 charges in a few minutes (fraudsters check the stolen card works),
3. 👯 a duplicate charge, and
4. 🪙 a late-night crypto top-up from abroad.

`np.random.default_rng(42)` makes the "random" data the same every time, so everyone in the room sees the same results.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
START, DAYS = pd.Timestamp("2026-07-01"), 92

# merchant: (min $, max $, visits per week, (earliest hour, latest hour), channel)
DAILY_MERCHANTS = {
    "KOPITIAM BEDOK":       (3.0, 8.0, 6, (7, 20), "card"),
    "BUS/MRT SIMPLYGO":     (1.2, 2.5, 8, (7, 22), "card"),
    "GRAB *TRANSPORT":      (8, 28, 3, (7, 23), "online"),
    "GRABFOOD":             (15, 40, 2, (11, 22), "online"),
    "NTUC FAIRPRICE":       (25, 140, 2, (9, 21), "card"),
    "STARBUCKS VIVOCITY":   (6.5, 9.5, 2, (8, 19), "card"),
    "SHOPEE SG":            (12, 90, 1, (10, 23), "online"),
    "GUARDIAN HEALTH":      (8, 45, 0.5, (10, 21), "card"),
    "GV CINEMAS":           (13, 30, 0.5, (12, 22), "card"),
    "UNIQLO ION":           (29, 120, 0.3, (11, 21), "card"),
}
# merchant: (amount, day of month); negative amount = money coming in
MONTHLY = {
    "SALARY ACME PTE LTD": (-4800.00, 25),
    "SP SERVICES":         (None, 5),   # utilities: varies each month
    "NETFLIX.COM":         (19.98, 12),
    "SPOTIFY":             (11.98, 15),
    "SINGTEL MOBILE":      (42.00, 8),
    "ANYTIME FITNESS":     (98.00, 1),
}

rows = []
for d in range(DAYS):
    day = START + pd.Timedelta(days=d)
    for m, (lo, hi, per_week, (h0, h1), channel) in DAILY_MERCHANTS.items():
        for _ in range(rng.poisson(per_week / 7)):
            ts = day + pd.Timedelta(hours=int(rng.integers(h0, h1)), minutes=int(rng.integers(0, 60)))
            rows.append((ts, m, round(float(rng.uniform(lo, hi)), 2), "SG", channel))
    for m, (amt, dom) in MONTHLY.items():
        if day.day == dom:
            amount = amt if amt is not None else round(float(rng.uniform(90, 160)), 2)
            rows.append((day + pd.Timedelta(hours=9), m, amount, "SG", "online"))

# 🚨 Plant the fraud patterns
fraud = [
    ("2026-08-14 02:47:00", "LUXEWATCH GENEVA", 3899.00, "CH", "online"),
    *[(f"2026-09-03 01:{12 + i * 2:02d}:00", "ONLINE STORE XJ7", 1.00, "GB", "online") for i in range(5)],
    ("2026-09-18 20:15:00", "SHOPEE SG", 64.90, "SG", "online"),
    ("2026-09-18 20:15:40", "SHOPEE SG", 64.90, "SG", "online"),
    ("2026-09-22 03:30:00", "CRYPTO-XCHANGE TOPUP", 2500.00, "MT", "online"),
]
rows += [(pd.Timestamp(t), m, a, c, ch) for t, m, a, c, ch in fraud]

df = (pd.DataFrame(rows, columns=["timestamp", "merchant", "amount", "country", "channel"])
        .sort_values("timestamp").reset_index(drop=True))
df.index.name = "txn_id"
print(f"{len(df)} transactions, {df.merchant.nunique()} merchants")
df.tail(8)

## Step 4: Categorise Merchants with One Batch LLM Call
**Coaching point on cost and speed:** we have ~400 transactions but only ~20 *unique* merchants. We send the unique names in **one** request and ask for a JSON list back, which is 1 API call instead of 400. With free-tier rate limits, this design decision decides whether your demo works.

`response_schema=list[MerchantCategory]` tells Gemini to return a list of objects.

In [ ]:
from typing import Literal
from pydantic import BaseModel

CATEGORIES = ["Food & Dining", "Groceries", "Transport", "Shopping", "Bills & Utilities",
              "Subscriptions", "Health & Fitness", "Entertainment", "Income", "Crypto & Investing", "Other"]

class MerchantCategory(BaseModel):
    merchant: str
    category: Literal[tuple(CATEGORIES)]

merchants = sorted(df.merchant.unique())
response = ask_gemini(
    "Categorise each Singapore card-statement merchant name. Return every merchant exactly as given.\n"
    + "\n".join(merchants),
    types.GenerateContentConfig(response_mime_type="application/json", response_schema=list[MerchantCategory]),
)
category_map = {item.merchant: item.category for item in response.parsed}
df["category"] = df.merchant.map(category_map).fillna("Other")

print(f"✅ Categorised {len(category_map)} merchants in 1 API call")
pd.Series(category_map, name="category").to_frame()

## Step 5: Detect Suspicious Transactions (Classic ML + Rules)
We **don't** ask the LLM to find fraud in 400 rows. It is slow, expensive, inconsistent and can't be audited. We use two tools that banks really use:

1. **Rules** for known patterns (duplicates, card testing, overseas at night). Easy to explain to a regulator.
2. **Isolation Forest** (an unsupervised ML model) to catch *unknown* weirdness. It learns what "normal" looks like and flags transactions that are easy to isolate from the rest.

First we turn each transaction into numeric **features** the model can use.

In [ ]:
from sklearn.ensemble import IsolationForest

spend = df[df.amount > 0].copy()
spend["hour"] = spend.timestamp.dt.hour
spend["is_foreign"] = (spend.country != "SG").astype(int)
spend["merchant_seen"] = spend.groupby("merchant").cumcount()                 # how many times seen before
spend["secs_since_same_merchant"] = (spend.groupby("merchant").timestamp.diff()
                                     .dt.total_seconds().fillna(7 * 86400))
spend["merchant_txns_last_hour"] = (spend.groupby("merchant", group_keys=False)[["timestamp", "amount"]]
                                    .apply(lambda g: g.rolling("1h", on="timestamp").amount.count()))

features = pd.DataFrame({
    "log_amount": np.log1p(spend.amount),
    "night": spend.hour.isin(range(0, 6)).astype(int),
    "is_foreign": spend.is_foreign,
    "new_merchant": (spend.merchant_seen == 0).astype(int),
    "log_secs_since_same": np.log1p(spend.secs_since_same_merchant),
})

model = IsolationForest(contamination=0.02, random_state=42).fit(features)
spend["ml_flag"] = model.predict(features) == -1

# Known-pattern rules
spend["rule"] = ""
spend.loc[(spend.secs_since_same_merchant < 300) & (spend.amount == spend.groupby("merchant").amount.shift()), "rule"] += "duplicate charge; "
spend.loc[spend.merchant_txns_last_hour >= 3, "rule"] += "card testing (many charges in 1 hour); "
spend.loc[(spend.is_foreign == 1) & spend.hour.isin(range(0, 6)), "rule"] += "overseas at night; "
spend.loc[(spend.merchant_seen == 0) & (spend.amount > 1000), "rule"] += "large first-time merchant; "

flagged = spend[spend.ml_flag | (spend.rule != "")]
print(f"🚩 {len(flagged)} of {len(spend)} spending transactions flagged")
flagged[["timestamp", "merchant", "amount", "country", "ml_flag", "rule"]]

### 🤔 Coaching discussion (5 min)
- Which planted frauds did the **rules** catch, and which did the **ML model** catch? Do they overlap?
- Did anything *innocent* get flagged (a **false positive**)? What does a false positive cost a bank (annoyed customer, blocked card) compared with a **false negative** (lost money)?
- Change `contamination` to `0.05`. What happens?

## Step 6: Explain Alerts in Plain English
*"Unusual activity detected"* gets ignored. *"Someone tried five $1 charges from a UK store at 1am. That's how fraudsters test stolen cards."* gets action.

The ML decided **what** is suspicious; the LLM explains **why it matters to the customer** and what to do.

In [ ]:
class Alert(BaseModel):
    txn_ids: list[int]
    risk: Literal["high", "medium", "low"]
    customer_message: str
    recommended_action: str

flag_text = flagged.reset_index()[["txn_id", "timestamp", "merchant", "amount", "country", "rule"]].to_string(index=False)
ALERT_PROMPT = f"""You are the fraud-alert writer for a Singapore bank app.
These card transactions were flagged by our detection system. Group related ones (e.g. repeated $1 charges) into one alert.
For each alert write a short, calm, specific message (max 2 sentences) explaining why it looks suspicious,
and one recommended action (e.g. "Freeze card in app", "Confirm if this was you", "Request refund from merchant").
Never ask for passwords or OTPs.

Customer usually spends in Singapore, mostly small amounts on food and transport.

Flagged transactions:
{flag_text}"""

alerts = ask_gemini(
    ALERT_PROMPT,
    types.GenerateContentConfig(response_mime_type="application/json", response_schema=list[Alert]),
).parsed

for a in alerts:
    print(f"[{a.risk.upper()}] txns {a.txn_ids}\n  💬 {a.customer_message}\n  👉 {a.recommended_action}\n")

## Step 7: The Money Coach (pandas calculates, LLM narrates)
LLMs are **bad at arithmetic** on long lists of numbers. They will confidently give you a wrong total. So we let **pandas** compute every number, then give the LLM a small, accurate summary to talk about.

In [ ]:
import matplotlib.pyplot as plt

legit = df.drop(index=flagged.index)  # leave suspicious items out of the budget
legit = legit.assign(month=legit.timestamp.dt.strftime("%Y-%m"))
monthly_spend = legit[legit.amount > 0].pivot_table(index="month", columns="category", values="amount", aggfunc="sum").fillna(0).round(2)
income = -legit[legit.amount < 0].groupby("month").amount.sum()
total_spend = monthly_spend.sum(axis=1)
recurring = (legit[legit.category.isin(["Subscriptions", "Health & Fitness", "Bills & Utilities"])]
             .groupby("merchant").amount.mean().round(2))

summary = f"""MONTHLY INCOME: {income.round(2).to_dict()}
MONTHLY SPENDING TOTAL: {total_spend.round(2).to_dict()}
SAVINGS RATE: {((income - total_spend) / income * 100).round(1).to_dict()} (percent)
SPENDING BY CATEGORY PER MONTH:
{monthly_spend.to_string()}
TOP 5 MERCHANTS BY TOTAL SPEND (3 months): {legit[legit.amount > 0].groupby("merchant").amount.sum().nlargest(5).round(2).to_dict()}
RECURRING PAYMENTS (average per charge): {recurring.to_dict()}
NUMBER OF KOPITIAM VISITS: {(legit.merchant == "KOPITIAM BEDOK").sum()}, GRABFOOD ORDERS: {(legit.merchant == "GRABFOOD").sum()}"""
print(summary)

fig, ax = plt.subplots(figsize=(8, 4))
monthly_spend.plot(kind="bar", stacked=True, ax=ax, colormap="tab20")
ax.set_title("Monthly spending by category"); ax.set_ylabel("SGD"); ax.legend(bbox_to_anchor=(1, 1), fontsize=8)
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

In [ ]:
COACH_PROMPT = """You are MoneyMate, a friendly, practical money coach for a young professional in Singapore.
Use ONLY the numbers in the financial summary below. Never calculate new totals yourself; if a number isn't
in the summary, say you don't have it. Keep answers under 120 words, use 1-3 bullet points of concrete tips.
You give budgeting tips only, not investment advice or product recommendations.

FINANCIAL SUMMARY:
{summary}

User: {question}"""

def coach(question):
    return ask_gemini(COACH_PROMPT.format(summary=summary, question=question)).text

print(coach("Where is most of my money going, and what's one easy way to save $200 a month?"))

### 🧪 Optional experiment: why not let the LLM do the maths?
Give Gemini 60 raw amounts and ask for the total. Compare it with pandas. Run it a few times; does the LLM's answer even stay the same?

In [ ]:
sample = df[df.amount > 0].amount.head(60)
llm_total = ask_gemini("Add up these numbers. Reply with only the total, nothing else:\n" + ", ".join(map(str, sample))).text
print(f"LLM says:    {llm_total.strip()}")
print(f"pandas says: {sample.sum():.2f}")

## Step 8: Build the Demo UI with Gradio 🏆
Two tabs: a **dashboard** (chart and fraud alerts) and a **chat coach**. Everything heavy was computed above, so the UI is instant.

In [ ]:
import gradio as gr

alert_rows = [[a.risk.upper(), ", ".join(map(str, a.txn_ids)), a.customer_message, a.recommended_action] for a in alerts]

def coach_chat(message, history):
    try:
        return coach(message)
    except Exception as e:
        return f"⚠️ Sorry, something went wrong: {e}"

with gr.Blocks(title="MoneyMate") as demo:
    gr.Markdown("# 💳 MoneyMate\nYour spending, explained. Suspicious activity, caught.")
    with gr.Tab("📊 Dashboard"):
        with gr.Row():
            gr.Number(value=float(income.mean().round(2)), label="Avg monthly income (SGD)")
            gr.Number(value=float(total_spend.mean().round(2)), label="Avg monthly spend (SGD)")
            gr.Number(value=float(((income - total_spend) / income * 100).mean().round(1)), label="Avg savings rate (%)")
        gr.Plot(value=fig, label="Monthly spending by category")
        gr.Markdown("### 🚨 Fraud alerts")
        gr.Dataframe(value=alert_rows, headers=["Risk", "Txn IDs", "What we noticed", "What to do"], wrap=True)
    with gr.Tab("💬 Money Coach"):
        gr.ChatInterface(
            fn=coach_chat,
            examples=["How much did I spend on food last month?", "Which subscriptions could I cut?", "Am I saving enough?"],
        )

# Launch it! (This will generate a public URL you can share with judges)
demo.launch(share=True, theme="soft")

## 🚀 Make It Your Own: Hackathon Extensions
- **Upload your own CSV:** add `gr.File` so judges can drop in a bank export (with fake data!).
- **Scam-message checker:** paste a suspicious SMS or WhatsApp message and get a scam verdict and explanation. Scams are a huge real problem.
- **Goals:** "I want to save $5,000 for Japan by June", then MoneyMate builds a budget and tracks progress.
- **SME version:** categorise business expenses, flag unusual supplier payments, draft GST summaries.
- **Better ML:** add more features (merchant risk score, device, distance from home) and compare models.

## 🎤 Pitching to Judges
- Show the **planted fraud being caught**, then the human-friendly alert. That's the "wow" moment.
- Explain the **hybrid architecture**: ML for detection, LLM for explanation, pandas for numbers. Fintech judges love seeing that you know the limits of LLMs.
- Mention **cost**: 1 API call to categorise everything, not 400.

## 🛡️ Responsible AI Checklist
- [ ] Synthetic data only. **Never put real bank data into the free Gemini tier**, which may use inputs to improve Google's products.
- [ ] Measure false positives; every wrongly blocked card is an angry customer.
- [ ] Fraud decisions are made by rules and models you can audit; the LLM only writes the message.
- [ ] The coach gives budgeting tips, not regulated investment advice.
- [ ] Look up the **MAS FEAT principles** (Fairness, Ethics, Accountability, Transparency) for AI in Singapore finance, and mention them in your pitch.